In [1]:
import os
import json

print(os.getcwd())

c:\Users\huyvu\OneDrive\Máy tính\data-pack


In [39]:
import json
PATH_CURRENT = os.getcwd()
PATH_EVAL = os.path.join(PATH_CURRENT, 'eval', 'E01.json')
PATH_HISTORY_INCIDENTS = os.path.join(PATH_CURRENT, 'incidents_history.json')
SAMPLE_EVAL = None
HISTORY_INCIDENTS = None
with open(PATH_EVAL, 'r', encoding='utf-8') as f:
    data = f.read()
    SAMPLE_EVAL = json.loads(data)

with open(PATH_HISTORY_INCIDENTS, 'r', encoding='utf-8') as f:
    data = f.read()
    HISTORY_INCIDENTS = json.loads(data)

print(f"""SAMPLE_EVAL
{'incident_id':20s} {SAMPLE_EVAL['incident_id']}
{'detected_at':20s} {SAMPLE_EVAL['detected_at']}
{'trigger_alert':20s} {SAMPLE_EVAL['trigger_alert']}
{'topology':20s} nodes({len(SAMPLE_EVAL['topology']['nodes'])}), edges({len(SAMPLE_EVAL['topology']['edges'])})
{'metrics_window':20s} from <{SAMPLE_EVAL['metrics_window']['from']}> to <{SAMPLE_EVAL['metrics_window']['to']}>
{'traces':20s} {len(SAMPLE_EVAL['traces'])} items
{'logs':20s} {len(SAMPLE_EVAL['logs'])} items
""")

SAMPLE_EVAL
incident_id          E01-2026-06-10-001
detected_at          2026-06-10T14:23:00Z
trigger_alert        {'service': 'checkout-svc', 'rule_id': 'latency-p99-high', 'severity': 'critical'}
topology             nodes(14), edges(17)
metrics_window       from <2026-06-10T13:53:00Z> to <2026-06-10T14:38:00Z>
traces               480 items
logs                 500 items



In [ ]:
def find_affected_services(incident: dict):
    affected = set()
    trigger = incident.get('trigger_alert', {})
    if trigger and 'service' in trigger:
        affected.add(trigger['service'])
    
    error_counts = {}
    for log in incident.get('logs', []):
        if log.get('level') in ('ERROR', 'FATAL'):
            svc = log.get('svc')
            if svc:
                error_counts[svc] = error_counts.get(svc, 0) + 1
    for svc, count in error_counts.items():
        if count >= 3:
            affected.add(svc)
    for trace in incident.get('traces', []):
        count = trace.get('count', 0)
        errors = trace.get('error_count', 0)
        if count > 0 and (errors / count) >= 0.05:
            affected.add(trace.get('from'))
            affected.add(trace.get('to'))
    return affected 
print(find_affected_services(SAMPLE_EVAL))

['payment-svc', 'checkout-svc']


In [ ]:
def get_all_historical_log_signatures(history: list[dict] = HISTORY_INCIDENTS):
    signatures = set()
    for incident in history:
        for sig in incident.get('log_signatures', []):
            signatures.add(sig)
    return sorted(list(signatures))
print(get_all_historical_log_signatures())

['429 returned to upstream', 'ConnectionPool: timeout acquiring connection', 'DB query latency > 5s on table', 'Failed to forward request: pool exhausted', 'GC pause: 4127ms (full GC, heap', 'OutOfMemoryError: Java heap space', 'Pod evicted: node out of memory', 'Retry exhausted after 5 attempts', 'TLS handshake failed: certificate has expired', 'cgroup OOM kill', 'consumer rebalance triggered', 'deadlock detected on table', 'degraded behavior detected', 'fallback failed, retrying request', 'feature distribution drift detected on field', 'lock timeout exceeded after', 'model inference confidence dropped below threshold', 'partition reassignment in progress', 'query took longer than threshold', 'rate limit exceeded for client', 'service error rate elevated', 'x509: certificate signed by unknown authority']


In [55]:
def match_log_signature(raw_msg: str, signature: str):
    parts = [p.strip().lower() for p in signature.split(':') if p.strip()]
    if not parts: return False
    msg_lower = raw_msg.lower()
    return all(part in msg_lower for part in parts)

In [58]:
ALL_SIGNATURES = get_all_historical_log_signatures()
def extract_log_features(logs: list[dict], all_signatures: list[str] = ALL_SIGNATURES):
    log_features = {sig: 0 for sig in all_signatures}
    for log in logs:
        msg = log.get('msg', '')
        for sig in all_signatures:
            if match_log_signature(msg, sig):
                log_features[sig] = 1
    return log_features
extract_log_features(SAMPLE_EVAL['logs'])

{'429 returned to upstream': 0,
 'ConnectionPool: timeout acquiring connection': 1,
 'DB query latency > 5s on table': 0,
 'Failed to forward request: pool exhausted': 1,
 'GC pause: 4127ms (full GC, heap': 0,
 'OutOfMemoryError: Java heap space': 0,
 'Pod evicted: node out of memory': 0,
 'Retry exhausted after 5 attempts': 0,
 'TLS handshake failed: certificate has expired': 0,
 'cgroup OOM kill': 0,
 'consumer rebalance triggered': 0,
 'deadlock detected on table': 0,
 'degraded behavior detected': 0,
 'fallback failed, retrying request': 0,
 'feature distribution drift detected on field': 0,
 'lock timeout exceeded after': 0,
 'model inference confidence dropped below threshold': 0,
 'partition reassignment in progress': 0,
 'query took longer than threshold': 0,
 'rate limit exceeded for client': 0,
 'service error rate elevated': 0,
 'x509: certificate signed by unknown authority': 0}

In [59]:
from datetime import datetime

def parse_ts(ts_str: str) -> datetime:
    return datetime.fromisoformat(ts_str.replace("Z", "+00:00"))

def extract_trace_features(traces: list[dict], detected_at_str: str) -> dict[tuple[str, str], dict]:
    detected_at = parse_ts(detected_at_str)
    
    edges_data = {}
    for t in traces:
        edge = (t["from"], t["to"])
        if edge not in edges_data:
            edges_data[edge] = {"baseline_p99": [], "anomaly_p99": [], "anomaly_errors": 0, "anomaly_count": 0}
            
        t_time = parse_ts(t["ts"])
        p99 = t.get("p99_ms", 0.0)
        
        if t_time < detected_at:
            edges_data[edge]["baseline_p99"].append(p99)
        else:
            edges_data[edge]["anomaly_p99"].append(p99)
            edges_data[edge]["anomaly_errors"] += t.get("error_count", 0)
            edges_data[edge]["anomaly_count"] += t.get("count", 0)
            
    features = {}
    for edge, data in edges_data.items():
        base_list = data["baseline_p99"]
        avg_base = sum(base_list) / len(base_list) if base_list else 100.0 # Giá trị mặc định nếu không có baseline
        
        anom_list = data["anomaly_p99"]
        avg_anom = sum(anom_list) / len(anom_list) if anom_list else avg_base
        
        deviation = (avg_anom / avg_base) if avg_base > 0 else 1.0
        
        err_rate = 0.0
        if data["anomaly_count"] > 0:
            err_rate = data["anomaly_errors"] / data["anomaly_count"]
            
        features[edge] = {
            "p99_deviation_ratio": round(deviation, 2),
            "error_rate": round(err_rate, 2)
        }
    return features
print(extract_trace_features(SAMPLE_EVAL['traces'], SAMPLE_EVAL['detected_at']))


{('edge-lb', 'checkout-svc'): {'p99_deviation_ratio': 0.98, 'error_rate': 0.0}, ('checkout-svc', 'payment-svc'): {'p99_deviation_ratio': 2.92, 'error_rate': 0.31}, ('payment-svc', 'payments-db'): {'p99_deviation_ratio': 0.97, 'error_rate': 0.0}, ('checkout-svc', 'cart-svc'): {'p99_deviation_ratio': 1.02, 'error_rate': 0.0}, ('cart-svc', 'cart-redis'): {'p99_deviation_ratio': 1.03, 'error_rate': 0.0}, ('edge-lb', 'search-svc'): {'p99_deviation_ratio': 0.96, 'error_rate': 0.0}}


In [60]:

def extract_metric_features(metrics_window: dict, detected_at_str: str) -> dict[tuple[str, str], dict]:
    detected_at = parse_ts(detected_at_str)
    samples = metrics_window.get("samples", {})
    
    features = {}
    for metric_key, points in samples.items():
        if not points:
            continue
            
        parts = metric_key.split(".")
        if len(parts) != 2:
            continue
        service, metric_name = parts[0], parts[1]
        
        baseline_vals = []
        anomaly_vals = []
        
        for ts_str, val in points:
            pt_time = parse_ts(ts_str)
            if pt_time < detected_at:
                baseline_vals.append(val)
            else:
                anomaly_vals.append(val)
                
        avg_base = sum(baseline_vals) / len(baseline_vals) if baseline_vals else 1.0
        avg_anom = sum(anomaly_vals) / len(anomaly_vals) if anomaly_vals else avg_base
        
        ratio = (avg_anom / avg_base) if avg_base > 0 else 1.0
        
        features[(service, metric_name)] = {
            "before": round(avg_base, 2),
            "after": round(avg_anom, 2),
            "ratio": round(ratio, 2)
        }
    return features
print(extract_metric_features(SAMPLE_EVAL['metrics_window'], SAMPLE_EVAL['detected_at']))

{('payment-svc', 'cpu'): {'before': 40.21, 'after': 60.64, 'ratio': 1.51}, ('payment-svc', 'latency_p99_ms'): {'before': 417.48, 'after': 980.66, 'ratio': 2.35}, ('checkout-svc', 'latency_p99_ms'): {'before': 178.97, 'after': 530.55, 'ratio': 2.96}}


In [61]:
def extract_features(incident: dict, history: list[dict]) -> dict:
    all_sigs = get_all_historical_log_signatures(history)
    detected_at = incident.get("detected_at", "")
    
    affected = find_affected_services(incident)
    logs_feat = extract_log_features(incident.get("logs", []), all_sigs)
    traces_feat = extract_trace_features(incident.get("traces", []), detected_at)
    metrics_feat = extract_metric_features(incident.get("metrics_window", {}), detected_at)
    
    return {
        "incident_id": incident.get("incident_id"),
        "trigger_alert": incident.get("trigger_alert", {}),
        "affected_services": affected,
        "log_features": logs_feat,
        "trace_features": traces_feat,
        "metric_features": metrics_feat
    }

print(extract_features(SAMPLE_EVAL, HISTORY_INCIDENTS))

{'incident_id': 'E01-2026-06-10-001', 'trigger_alert': {'service': 'checkout-svc', 'rule_id': 'latency-p99-high', 'severity': 'critical'}, 'affected_services': ['payment-svc', 'checkout-svc'], 'log_features': {'429 returned to upstream': 0, 'ConnectionPool: timeout acquiring connection': 1, 'DB query latency > 5s on table': 0, 'Failed to forward request: pool exhausted': 1, 'GC pause: 4127ms (full GC, heap': 0, 'OutOfMemoryError: Java heap space': 0, 'Pod evicted: node out of memory': 0, 'Retry exhausted after 5 attempts': 0, 'TLS handshake failed: certificate has expired': 0, 'cgroup OOM kill': 0, 'consumer rebalance triggered': 0, 'deadlock detected on table': 0, 'degraded behavior detected': 0, 'fallback failed, retrying request': 0, 'feature distribution drift detected on field': 0, 'lock timeout exceeded after': 0, 'model inference confidence dropped below threshold': 0, 'partition reassignment in progress': 0, 'query took longer than threshold': 0, 'rate limit exceeded for client